## Matplotlib M+L Flaws Robustness Study

In [ ]:
# === Robustness scoring (streaming/resumable) ===
# Appends one row at a time to output CSV so you keep progress even if it crashes.

import os, ast, re, sys, time, hashlib
import pandas as pd

# ---------- CONFIG ----------
SRC = 'github_matplotlib_audit_1'  # base name or .csv
LLM_MIN_INTERVAL = float(os.getenv("LLM_MIN_INTERVAL", "6.0"))  # throttle between LLM calls (sec)
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
MAX_ROWS = int(os.getenv("MAX_ROWS", "0"))  # 0 = all rows; else stop early for testing
PRINT_EVERY = int(os.getenv("PRINT_EVERY", "50"))  # progress print cadence

# === NEW: sampling config ===
SAMPLE_N = int(os.getenv("SAMPLE_N", "638"))     # set 0 to disable sampling
SAMPLE_SEED = int(os.getenv("SAMPLE_SEED", "1337"))

_LLM_LAST = 0.0  # do not touch

# ---------- Gemini setup (guarded) ----------
HAVE_GEMINI = False
try:
    import google.generativeai as genai
    api_key = os.getenv("GOOGLE_API_KEY", None)
    if not api_key:
        from GEMINI_API_KEY import GEMINI_API_KEY as api_key  # optional fallback
    if not api_key:
        raise RuntimeError("No Gemini API key found in env or GEMINI_API_KEY.py")
    genai.configure(api_key=api_key)
    client = genai.GenerativeModel(MODEL_NAME)
    HAVE_GEMINI = True
except Exception as e:
    print(f"⚠️ Gemini disabled: {e}")
    HAVE_GEMINI = False

# ---------- Non-contextual rules ----------
def find_noncontextual_flaws(mpl_file: str):
    flaws = []
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return flaws
    scan_text = mpl_file[mpl_index:]

    # Descriptive labels
    for fn in ["title", "xlabel", "ylabel"]:
        match = re.search(rf"{fn}\s*\(\s*['\"]([^'\"]*)['\"]", scan_text)
        if match:
            txt = match.group(1).strip().lower()
            if not txt or txt in ["x", "y", "series 1"]:
                flaws.append(f"MISSING_{fn.upper()}")
        else:
            flaws.append(f"MISSING_{fn.upper()}")

    # Legend required when multiple series
    plot_count = len(re.findall(r'plot\s*\(', scan_text))
    scatter_count = len(re.findall(r'scatter\s*\(', scan_text))
    if (plot_count + scatter_count > 1) and 'legend(' not in scan_text:
        flaws.append("MISSING_LEGEND")

    # Font size
    font_matches = re.findall(r'fontsize\s*=\s*(\d+)', scan_text)
    if any(int(size) < 15 for size in font_matches):
        flaws.append("FONTSIZE_TOO_SMALL")

    # Figure size
    fig_match = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_match:
        w, h = float(fig_match.group(1)), float(fig_match.group(2))
        if w < 8 or h < 5:
            flaws.append("FIGSIZE_TOO_SMALL")

    # High-contrast colors
    color_matches = re.findall(r'color\s*=\s*[\'"]([^\'"]+)[\'"]', scan_text)
    safe_colors = {
        "#000000", "#0072B2", "#009E73", "#D55E00",
        "black", "blue", "green", "orange"
    }
    if any(color.lower() not in safe_colors for color in color_matches):
        flaws.append("INSUFFICIENT_COLOR_CONTRAST")

    # No animations
    if "FuncAnimation" in scan_text or "animation." in scan_text:
        flaws.append("ANIMATIONS")

    # Inverted Y-axis
    if re.search(r'\.\s*invert_yaxis\s*\(', scan_text):
        flaws.append("INVERTED_Y_AXIS")

    # Truncated Y-axis
    for m in re.finditer(r'(?:set_)?ylim\s*\(\s*([\-]?\d+(?:\.\d+)?)\s*,', scan_text):
        lower = float(m.group(1))
        if abs(lower) > 1e-6:
            flaws.append("TRUNCATED_Y_AXIS")
            break

    # 3D effects
    if (re.search(r'["\']\s*3d\s*["\']', scan_text) or
        "Axes3D" in scan_text or
        "plot_surface(" in scan_text):
        flaws.append("3D_EFFECTS")

    # Aspect ratio
    fig_aspect = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_aspect:
        w, h = float(fig_aspect.group(1)), float(fig_aspect.group(2))
        if h != 0:
            ratio = w / h
            if ratio < 0.5 or ratio > 2.0:
                flaws.append("TAMPERED_ASPECT_RATIO")
    if re.search(r'set_aspect\s*\(|aspect\s*=', scan_text):
        flaws.append("TAMPERED_ASPECT_RATIO")

    # Dual Y
    if re.search(r'twin[xy]\s*\(', scan_text) or 'secondary_y=True' in scan_text:
        flaws.append("DUAL_Y_AXES")

    return list(sorted(set(flaws)))

# ---------- Contextual rules (your prompt kept) ----------
def generate_response(prompt: str) -> str:
    result = client.generate_content(prompt)
    candidate = result.candidates[0]
    return candidate.content.parts[0].text.strip()

_CONTEXTUAL_RULES_PROMPT_TEMPLATE = """You are an expert in data visualization integrity. I will provide you with:

1. A list of misleading visualization rules (each with a RULE_CODE and description),
2. A Matplotlib code snippet that generates a chart.

Your task:
- Analyze the code and detect which rules are violated based solely on what can be inferred from the code itself (e.g., axis behavior, titles, aspect ratio, annotations).
- Output only the list of violated RULE_CODEs in exactly this format: ["RULE_CODE1", "RULE_CODE2", ...]
- If the graph does not violate any rules, return: NONE

Rule Codes and Descriptions:

BIASED_TITLE:
A graph uses a biased or emotionally slanted title that influences interpretation before data is analyzed.

MISLEADING_ANNOTATIONS:
Annotations suggest causality or relationships that are not statistically or contextually justified.

DECEPTIVE_LABELS:
Y-axis or x-axis labels are vague, reversed, or omit key categories, leading to confusion.

FRAMING_BIAS:
External context or textual framing (e.g., comments, hashtags, plot subtitles) introduces bias not reflected in the graph.

INVERTED_AXES:
Y-axis is reversed (top to bottom), which misleads users by flipping the meaning of increases/decreases.

TRUNCATED_AXES:
Y-axis does not start at zero, which exaggerates visual differences.

ASPECT_RATIO_DISTORTION:
Aspect ratio is altered (e.g., too stretched or squished), making trends look steeper or flatter than they are.

DUAL_AXES:
Chart uses two different y-axes that may falsely suggest correlation between unrelated data series.

NON_SEQUENTIAL_AXIS:
X or Y axis uses a non-logical or out-of-order sequence (e.g., age ranges like 18-34, 45-55, 35-44).

Matplotlib Code:
{code}
"""

def find_contextual_flaws(mpl_file: str):
    # Bail if no matplotlib
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return []
    scan_text = mpl_file[mpl_index:]

    if not HAVE_GEMINI:
        return []

    # Throttle one call at a time
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(LLM_MIN_INTERVAL - elapsed)

    prompt = _CONTEXTUAL_RULES_PROMPT_TEMPLATE.format(code=scan_text)
    try:
        raw = generate_response(prompt)
        _LLM_LAST = time.time()
    except Exception as e:
        # Timeouts / 429 / transient network: skip this row so we keep moving
        msg = str(e)
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("Timeout" in msg) or ("timed out" in msg.lower()):
            # small backoff
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return []
        print(f"⚠️ Gemini error: {e}")
        return []

    if raw.strip().upper() == "NONE":
        return []

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return list(sorted(set(str(x).strip().upper() for x in parsed)))
    except Exception:
        # one more shot with stricter instruction
        try:
            raw2 = generate_response(prompt + '\nReturn ONLY a valid Python list literal of strings like ["RULE","RULE2"].')
            if raw2.strip().upper() == "NONE":
                return []
            parsed2 = ast.literal_eval(raw2)
            if isinstance(parsed2, list):
                return list(sorted(set(str(x).strip().upper() for x in parsed2)))
        except Exception:
            return []
    return []

# ---------- Rule columns ----------
NONCONTEXTUAL_CODES = [
    "MISSING_TITLE","MISSING_XLABEL","MISSING_YLABEL","MISSING_LEGEND",
    "FONTSIZE_TOO_SMALL","FIGSIZE_TOO_SMALL","INSUFFICIENT_COLOR_CONTRAST",
    "ANIMATIONS","INVERTED_Y_AXIS","TRUNCATED_Y_AXIS","3D_EFFECTS",
    "TAMPERED_ASPECT_RATIO","DUAL_Y_AXES",
]
CONTEXTUAL_CODES = [
    "BIASED_TITLE","MISLEADING_ANNOTATIONS","DECEPTIVE_LABELS","FRAMING_BIAS",
    "INVERTED_AXES","TRUNCATED_AXES","ASPECT_RATIO_DISTORTION","DUAL_AXES",
    "NON_SEQUENTIAL_AXIS",
]
ALL_RULE_COLS = NONCONTEXTUAL_CODES + CONTEXTUAL_CODES

# === NEW: natural language descriptions for all rule codes (fed to Gemini summaries only; we DO NOT pass Matplotlib code) ===
RULE_DESCRIPTIONS = {
    # Non-contextual
    "MISSING_TITLE": "The chart has no title.",
    "MISSING_XLABEL": "The x-axis lacks a descriptive label.",
    "MISSING_YLABEL": "The y-axis lacks a descriptive label.",
    "MISSING_LEGEND": "Multiple series are shown but there is no legend.",
    "FONTSIZE_TOO_SMALL": "Text is small and may be hard to read.",
    "FIGSIZE_TOO_SMALL": "Figure size is small, reducing readability.",
    "INSUFFICIENT_COLOR_CONTRAST": "Colors may not have enough contrast for visibility.",
    "ANIMATIONS": "Motion/animations can distract or be inaccessible.",
    "INVERTED_Y_AXIS": "The y-axis is inverted, flipping increases/decreases.",
    "TRUNCATED_Y_AXIS": "The y-axis baseline is truncated (not starting at zero).",
    "3D_EFFECTS": "3D styling can distort visual perception.",
    "TAMPERED_ASPECT_RATIO": "Aspect ratio may exaggerate or flatten trends.",
    "DUAL_Y_AXES": "Two y-axes can imply false relationships.",
    # Contextual
    "BIASED_TITLE": "Title uses biased or emotive language.",
    "MISLEADING_ANNOTATIONS": "Annotations imply unjustified relationships/causality.",
    "DECEPTIVE_LABELS": "Axis labels are vague, reversed, or omit categories.",
    "FRAMING_BIAS": "Framing text introduces bias not supported by the plot.",
    "INVERTED_AXES": "Axis direction is reversed.",
    "TRUNCATED_AXES": "Axis range is truncated, exaggerating differences.",
    "ASPECT_RATIO_DISTORTION": "Stretched/squished aspect ratio alters perceived slope.",
    "DUAL_AXES": "Two y-axes may falsely suggest correlation.",
    "NON_SEQUENTIAL_AXIS": "Axis categories are out of logical order."
}

# === NEW: summary prompts (NO Matplotlib code is included) ===
_SUMMARY_PROMPT_TEMPLATE = """You are writing a concise, screen-reader-friendly chart description for blind/low-vision users.
You will be given a list of observed violations (codes + plain-language descriptions). You DO NOT see the chart or its code.
Task:
1) In 1–2 sentences, give a neutral, generic description of what such a chart might convey (avoid specifics you don't know).
2) In 1–3 sentences, describe the issues in plain language based ONLY on the provided violations. Do not speculate beyond them.
3) Keep it under 120 words. No emojis, no markdown, no code words; write for end users, not developers.
{extra}
Violations to consider:
{violations}
"""

def _throttle():
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(LLM_MIN_INTERVAL - elapsed)

def _safe_gemini_call(prompt: str) -> str:
    if not HAVE_GEMINI:
        return ""
    try:
        _throttle()
        txt = generate_response(prompt)
        # timestamp for next throttle window
        global _LLM_LAST
        _LLM_LAST = time.time()
        return txt
    except Exception as e:
        msg = str(e)
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("Timeout" in msg) or ("timed out" in msg.lower()):
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return ""
        print(f"⚠️ Gemini summary error: {e}")
        return ""

def _codes_to_lines(codes):
    # turn ["A","B"] into ["A: desc", "B: desc"]
    return [f"{c}: {RULE_DESCRIPTIONS.get(c, c)}" for c in codes]

def generate_summary_from_rules(codes, withhold=None) -> str:
    # Build the violations list we actually show the model (omit withheld)
    visible_codes = [c for c in codes if c != withhold]
    if not visible_codes:
        return ""  # nothing to say; skip LLM call
    violations_text = "\n".join(_codes_to_lines(visible_codes))
    extra = ""
    if withhold:
        # Make it explicit that it must not hint at the withheld item
        extra = f"CRITICAL: Do NOT mention or allude to this violation: {withhold}: {RULE_DESCRIPTIONS.get(withhold, withhold)}.\n"
    prompt = _SUMMARY_PROMPT_TEMPLATE.format(violations=violations_text, extra=extra)
    return _safe_gemini_call(prompt)

# === NEW: deterministic picker for which flaw to withhold per row ===
def pick_masked_rule(codes, row_id: int) -> str:
    """Pick one code to hide using a stable pseudo-random mapping of row_id."""
    if not codes:
        return ""
    # stable index based on row_id
    digest = hashlib.md5(f"mask::{row_id}".encode("utf-8")).hexdigest()
    idx = int(digest, 16) % len(codes)
    return list(sorted(set(codes)))[idx]

# ---------- Load input ----------
fname = SRC if os.path.exists(SRC) else (SRC + '.csv' if os.path.exists(SRC + '.csv') else SRC)
try:
    df = pd.read_csv(fname)
except Exception:
    df = pd.read_csv(SRC + '.csv')
    fname = SRC + '.csv'

print(f"Loaded: {fname}  (rows={len(df)})")

# Detect code column
POSSIBLE_CODE_COLS = ["matplotlib_code","code","file_content","content","snippet","source","body","text"]
code_col = None
for c in POSSIBLE_CODE_COLS:
    if c in df.columns:
        code_col = c; break
if code_col is None:
    for c in df.columns:
        if df[c].dtype == object:
            sample = df[c].dropna().astype(str).head(10)
            if sample.str.contains("import matplotlib|from matplotlib", regex=True, case=False).any():
                code_col = c; break
if code_col is None:
    raise ValueError("Could not find a column with Matplotlib code. Add a column named e.g., 'matplotlib_code'.")

print(f"Using code column: {code_col}")

# === NEW: sample down to N rows (deterministic) ===
if SAMPLE_N and SAMPLE_N > 0 and SAMPLE_N < len(df):
    df = df.sample(n=SAMPLE_N, random_state=SAMPLE_SEED).sort_index()
    print(f"Sampling to {len(df)} rows (SAMPLE_N={SAMPLE_N}, SEED={SAMPLE_SEED})")

# ---------- Prepare streaming output ----------
base, ext = os.path.splitext(fname)
# === NEW: make output name reflect sampling to avoid clobbering a full-run file
suffix = f"_n{len(df)}" if SAMPLE_N and SAMPLE_N > 0 and SAMPLE_N < len(pd.read_csv(fname, nrows=1))*0 + 10**6 else ""
out = f"{base}_scored_stream{suffix}{ext or '.csv'}"

# Build a unified header: row_id + original cols + rule flags + list cols
orig_cols = list(df.columns)
header_cols = (["row_id"] + orig_cols +
               [c for c in ALL_RULE_COLS if c not in orig_cols] +
               ["NONCONTEXTUAL_LIST","CONTEXTUAL_LIST"] +
               # === NEW summary columns ===
               ["MASKED_RULE","SUMMARY_ALL","SUMMARY_MISSING_ONE"])

# Resume support: find already processed row_ids
processed_ids = set()
if os.path.exists(out):
    try:
        prev = pd.read_csv(out, usecols=["row_id"])
        processed_ids = set(prev["row_id"].dropna().astype(int).tolist())
        print(f"Resuming; already have {len(processed_ids)} rows in {out}")
    except Exception as e:
        print(f"Note: couldn't read existing {out} for resume ({e}); starting fresh.")

# Write header if new file
write_header = not os.path.exists(out)
if write_header:
    pd.DataFrame(columns=header_cols).to_csv(out, index=False)
    write_header = False  # header is now written

# ---------- Main loop (streaming) ----------
def score_row(code: str):
    if not isinstance(code, str):
        return [], []
    nonctx = find_noncontextual_flaws(code)
    ctx = find_contextual_flaws(code) if HAVE_GEMINI else []
    return nonctx, ctx

rows_done = 0
for idx, row in df.iterrows():
    row_id = int(idx)
    if row_id in processed_ids:
        continue

    code = row[code_col]
    nonctx_list, ctx_list = score_row(code)

    # Build a single-row dict with original data + flags
    out_row = dict(row)  # copies original columns
    out_row["row_id"] = row_id
    for r in NONCONTEXTUAL_CODES:
        out_row[r] = 1 if r in nonctx_list else 0
    for r in CONTEXTUAL_CODES:
        out_row[r] = 1 if r in ctx_list else 0
    out_row["NONCONTEXTUAL_LIST"] = ";".join(nonctx_list)
    out_row["CONTEXTUAL_LIST"] = ";".join(ctx_list)

    # === NEW: summaries built ONLY from rule codes (no Matplotlib code passed to Gemini) ===
    detected_all = sorted(set(nonctx_list + ctx_list))
    masked_rule = pick_masked_rule(detected_all, row_id) if detected_all else ""
    summary_all = ""
    summary_missing = ""
    if HAVE_GEMINI and detected_all:
        # full list summary
        summary_all = generate_summary_from_rules(detected_all, withhold=None)
        # withhold one violation (do not show it to the model)
        summary_missing = generate_summary_from_rules(detected_all, withhold=masked_rule)

    out_row["MASKED_RULE"] = masked_rule
    out_row["SUMMARY_ALL"] = summary_all
    out_row["SUMMARY_MISSING_ONE"] = summary_missing

    # Append to CSV immediately
    pd.DataFrame([out_row])[header_cols].to_csv(out, mode="a", header=False, index=False)

    rows_done += 1
    if rows_done % PRINT_EVERY == 0:
        print(f"Processed {rows_done} rows (last row_id={row_id}) → {out}")

    if MAX_ROWS and rows_done >= MAX_ROWS:
        print(f"Hit MAX_ROWS={MAX_ROWS}; stopping early.")
        break

print(f"✅ Done. Appended {rows_done} new rows → {out}")


C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: github_matplotlib_audit_1.csv  (rows=14956)
Using code column: code
Sampling to 638 rows (SAMPLE_N=638, SEED=1337)
Processed 50 rows (last row_id=1149) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 100 rows (last row_id=2428) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 150 rows (last row_id=3577) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 200 rows (last row_id=4550) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 250 rows (last row_id=5666) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 300 rows (last row_id=6713) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 350 rows (last row_id=7908) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 400 rows (last row_id=9108) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 450 rows (last row_id=10430) → github_matplotlib_audit_1_scored_stream_n638.csv
Processed 500 rows (last row_id=11611) → github_matplotlib_audit_1_scored_st

In [5]:
df1 = pd.read_csv("github_matplotlib_audit_1_scored_stream_n638.csv")


In [6]:
df1.head()

,row_id,repo,filename,pushed_date,code,MISSING_TITLE,MISSING_XLABEL,MISSING_YLABEL,MISSING_LEGEND,FONTSIZE_TOO_SMALL,...,INVERTED_AXES,TRUNCATED_AXES,ASPECT_RATIO_DISTORTION,DUAL_AXES,NON_SEQUENTIAL_AXIS,NONCONTEXTUAL_LIST,CONTEXTUAL_LIST,MASKED_RULE,SUMMARY_ALL,SUMMARY_MISSING_ONE
0,22,jynrock/matplotlibNFT,voxels.py,2022-01-05,"""""""\n==========================\n3D voxel / vo...",1,1,1,0,0,...,0,0,0,0,0,3D_EFFECTS;INSUFFICIENT_COLOR_CONTRAST;MISSING...,NaN,MISSING_YLABEL,NaN,NaN
1,31,v1tal303/HM-land-registry-data,land-data.py,2022-01-07,import pandas as pd\nimport numpy as np\nimpor...,1,1,1,1,0,...,0,0,0,0,0,MISSING_LEGEND;MISSING_TITLE;MISSING_XLABEL;MI...,NaN,MISSING_XLABEL,NaN,NaN
2,34,rahulumrao/Matplotlib_pyplot_script,1_plot.py,2022-01-07,#!/usr/env/python\n# Import package\nimport nu...,1,1,1,0,0,...,0,0,0,0,0,INSUFFICIENT_COLOR_CONTRAST;MISSING_TITLE;MISS...,NaN,INSUFFICIENT_COLOR_CONTRAST,NaN,NaN
3,48,pythymcpyface/CryptoAppPython,statistical_analysis.py,2022-01-09,import datetime\nimport multiprocessing as mp\...,0,0,0,0,0,...,0,0,0,0,0,TRUNCATED_Y_AXIS,NaN,TRUNCATED_Y_AXIS,NaN,NaN
4,53,mmoraschini/annoplot,README.md,2022-01-09,# annoplot\nThis module lets you draw clickabl...,1,1,1,1,0,...,0,0,0,0,0,MISSING_LEGEND;MISSING_TITLE;MISSING_XLABEL;MI...,NaN,MISSING_YLABEL,NaN,NaN


In [9]:
# === Robustness scoring (streaming/resumable) + dual summaries & similarity ===
# Appends one row at a time to output CSV so you keep progress even if it crashes.

import os, ast, re, sys, time, hashlib, random
import pandas as pd

# ---------- CONFIG ----------
SRC = 'github_matplotlib_audit_1'  # base name or .csv
LLM_MIN_INTERVAL = float(os.getenv("LLM_MIN_INTERVAL", "6.0"))  # throttle between LLM calls (sec)
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
MAX_ROWS = int(os.getenv("MAX_ROWS", "0"))  # 0 = all rows; else stop early for testing
PRINT_EVERY = int(os.getenv("PRINT_EVERY", "50"))  # progress print cadence
SAMPLE_N = int(os.getenv("SAMPLE_N", "10"))  # random sample size; set 0 to use all
SIM_URL = os.getenv("SIM_URL", "http://127.0.0.1:8002/compute_semantic_similarity")

_LLM_LAST = 0.0  # do not touch

# ---------- Gemini setup (guarded) ----------
HAVE_GEMINI = False
try:
    import google.generativeai as genai
    api_key = os.getenv("GOOGLE_API_KEY", None)
    if not api_key:
        from GEMINI_API_KEY import GEMINI_API_KEY as api_key  # optional fallback
    if not api_key:
        raise RuntimeError("No Gemini API key found in env or GEMINI_API_KEY.py")
    genai.configure(api_key=api_key)
    client = genai.GenerativeModel(MODEL_NAME)
    HAVE_GEMINI = True
except Exception as e:
    print(f"⚠️ Gemini disabled: {e}")
    HAVE_GEMINI = False

# ---------- Optional HTTP client for similarity service ----------
try:
    import requests
    HAVE_REQUESTS = True
except Exception:
    HAVE_REQUESTS = False

# ---------- Rule dictionaries (for masking instruction text) ----------
NONCONTEXTUAL_CODES = [
    "MISSING_TITLE","MISSING_XLABEL","MISSING_YLABEL","MISSING_LEGEND",
    "FONTSIZE_TOO_SMALL","FIGSIZE_TOO_SMALL","INSUFFICIENT_COLOR_CONTRAST",
    "ANIMATIONS","INVERTED_Y_AXIS","TRUNCATED_Y_AXIS","3D_EFFECTS",
    "TAMPERED_ASPECT_RATIO","DUAL_Y_AXES",
]
CONTEXTUAL_CODES = [
    "BIASED_TITLE","MISLEADING_ANNOTATIONS","DECEPTIVE_LABELS","FRAMING_BIAS",
    "INVERTED_AXES","TRUNCATED_AXES","ASPECT_RATIO_DISTORTION","DUAL_AXES",
    "NON_SEQUENTIAL_AXIS",
]
ALL_RULE_COLS = NONCONTEXTUAL_CODES + CONTEXTUAL_CODES

# Short, targeted “do not mention” instructions for each violation
RULE_DESCRIPTIONS = {
    # non-contextual
    "MISSING_TITLE": "Do NOT mention titles or the presence/absence/quality of a title.",
    "MISSING_XLABEL": "Do NOT mention x-axis labels or naming of the x-axis.",
    "MISSING_YLABEL": "Do NOT mention y-axis labels or naming of the y-axis.",
    "MISSING_LEGEND": "Do NOT mention legends or multiple series identification by legend.",
    "FONTSIZE_TOO_SMALL": "Do NOT mention font size, readability, or text being small/hard to read.",
    "FIGSIZE_TOO_SMALL": "Do NOT mention figure size, resolution, or that the chart is small.",
    "INSUFFICIENT_COLOR_CONTRAST": "Do NOT mention colors, contrast, or color-blind accessibility.",
    "ANIMATIONS": "Do NOT mention animations, motion, or anything moving/changing.",
    "INVERTED_Y_AXIS": "Do NOT mention an inverted y-axis or directionality (up vs down).",
    "TRUNCATED_Y_AXIS": "Do NOT mention where the y-axis starts or truncated axes.",
    "3D_EFFECTS": "Do NOT mention 3D effects or 3D styling.",
    "TAMPERED_ASPECT_RATIO": "Do NOT mention aspect ratio, stretching, or squashing.",
    "DUAL_Y_AXES": "Do NOT mention multiple y-axes, secondary axes, or dual scales.",
    # contextual
    "BIASED_TITLE": "Do NOT mention bias in the title or emotionally slanted wording.",
    "MISLEADING_ANNOTATIONS": "Do NOT mention annotations implying causality or unjustified relationships.",
    "DECEPTIVE_LABELS": "Do NOT mention vague/missing/misleading axis labels.",
    "FRAMING_BIAS": "Do NOT include external textual framing beyond what the chart shows.",
    "INVERTED_AXES": "Do NOT mention inverted axes or directionality.",
    "TRUNCATED_AXES": "Do NOT mention whether an axis starts at zero or truncated axes.",
    "ASPECT_RATIO_DISTORTION": "Do NOT mention aspect ratio distortion or stretching.",
    "DUAL_AXES": "Do NOT mention dual axes or two y-axes.",
    "NON_SEQUENTIAL_AXIS": "Do NOT mention non-sequential or out-of-order axis categories.",
}

# ---------- Non-contextual rules ----------
def find_noncontextual_flaws(mpl_file: str):
    flaws = []
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return flaws
    scan_text = mpl_file[mpl_index:]

    # Descriptive labels
    for fn in ["title", "xlabel", "ylabel"]:
        match = re.search(rf"{fn}\s*\(\s*['\"]([^'\"]*)['\"]", scan_text)
        if match:
            txt = match.group(1).strip().lower()
            if not txt or txt in ["x", "y", "series 1"]:
                flaws.append(f"MISSING_{fn.upper()}")
        else:
            flaws.append(f"MISSING_{fn.upper()}")

    # Legend required when multiple series
    plot_count = len(re.findall(r'plot\s*\(', scan_text))
    scatter_count = len(re.findall(r'scatter\s*\(', scan_text))
    if (plot_count + scatter_count > 1) and 'legend(' not in scan_text:
        flaws.append("MISSING_LEGEND")

    # Font size
    font_matches = re.findall(r'fontsize\s*=\s*(\d+)', scan_text)
    if any(int(size) < 15 for size in font_matches):
        flaws.append("FONTSIZE_TOO_SMALL")

    # Figure size
    fig_match = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_match:
        w, h = float(fig_match.group(1)), float(fig_match.group(2))
        if w < 8 or h < 5:
            flaws.append("FIGSIZE_TOO_SMALL")

    # High-contrast colors (very rough heuristic)
    color_matches = re.findall(r'color\s*=\s*[\'"]([^\'"]+)[\'"]', scan_text)
    safe_colors = {
        "#000000", "#0072B2", "#009E73", "#D55E00",
        "black", "blue", "green", "orange"
    }
    if any(str(color).lower() not in safe_colors for color in color_matches):
        flaws.append("INSUFFICIENT_COLOR_CONTRAST")

    # No animations
    if "FuncAnimation" in scan_text or "animation." in scan_text:
        flaws.append("ANIMATIONS")

    # Inverted Y-axis
    if re.search(r'\.\s*invert_yaxis\s*\(', scan_text):
        flaws.append("INVERTED_Y_AXIS")

    # Truncated Y-axis
    for m in re.finditer(r'(?:set_)?ylim\s*\(\s*([\-]?\d+(?:\.\d+)?)\s*,', scan_text):
        lower = float(m.group(1))
        if abs(lower) > 1e-6:
            flaws.append("TRUNCATED_Y_AXIS")
            break

    # 3D effects
    if (re.search(r'["\']\s*3d\s*["\']', scan_text) or
        "Axes3D" in scan_text or
        "plot_surface(" in scan_text):
        flaws.append("3D_EFFECTS")

    # Aspect ratio
    fig_aspect = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_aspect:
        w, h = float(fig_aspect.group(1)), float(fig_aspect.group(2))
        if h != 0:
            ratio = w / h
            if ratio < 0.5 or ratio > 2.0:
                flaws.append("TAMPERED_ASPECT_RATIO")
    if re.search(r'set_aspect\s*\(|aspect\s*=', scan_text):
        flaws.append("TAMPERED_ASPECT_RATIO")

    # Dual Y
    if re.search(r'twin[xy]\s*\(', scan_text) or 'secondary_y=True' in scan_text:
        flaws.append("DUAL_Y_AXES")

    return list(sorted(set(flaws)))

# ---------- Contextual rule detector via Gemini ----------
def generate_llm(text_prompt: str) -> str:
    """Low-level Gemini call with throttling; returns plain text (single line)."""
    if not HAVE_GEMINI:
        return ""
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(LLM_MIN_INTERVAL - elapsed)
    try:
        result = client.generate_content(text_prompt)
        _LLM_LAST = time.time()
        candidate = result.candidates[0]
        out = candidate.content.parts[0].text if candidate and candidate.content and candidate.content.parts else ""
        return re.sub(r"\s+", " ", (out or "").strip())
    except Exception as e:
        # transient failures: skip
        msg = str(e)
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("Timeout" in msg) or ("timed out" in msg.lower()):
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return ""
        return ""

_CONTEXTUAL_RULES_PROMPT_TEMPLATE = """You are an expert in data visualization integrity. I will provide you with:
1) A list of misleading visualization rules (each with a RULE_CODE and description),
2) A Matplotlib code snippet that generates a chart.

Your task:
- Analyze the code and detect which rules are violated based solely on what can be inferred from the code itself (e.g., axis behavior, titles, aspect ratio, annotations).
- Output only the list of violated RULE_CODEs in exactly this format: ["RULE_CODE1", "RULE_CODE2", ...]
- If the graph does not violate any rules, return: NONE

Rule Codes and Descriptions:

BIASED_TITLE:
A graph uses a biased or emotionally slanted title that influences interpretation before data is analyzed.

MISLEADING_ANNOTATIONS:
Annotations suggest causality or relationships that are not statistically or contextually justified.

DECEPTIVE_LABELS:
Y-axis or x-axis labels are vague, reversed, or omit key categories, leading to confusion.

FRAMING_BIAS:
External context or textual framing (e.g., comments, hashtags, plot subtitles) introduces bias not reflected in the graph.

INVERTED_AXES:
Y-axis is reversed (top to bottom), which misleads users by flipping the meaning of increases/decreases.

TRUNCATED_AXES:
Y-axis does not start at zero, which exaggerates visual differences.

ASPECT_RATIO_DISTORTION:
Aspect ratio is altered (e.g., too stretched or squished), making trends look steeper or flatter than they are.

DUAL_AXES:
Chart uses two different y-axes that may falsely suggest correlation between unrelated data series.

NON_SEQUENTIAL_AXIS:
X or Y axis uses a non-logical or out-of-order sequence (e.g., age ranges like 18-34, 45-55, 35-44).

Matplotlib Code:
{code}
"""

def find_contextual_flaws(mpl_file: str):
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1 or not HAVE_GEMINI:
        return []
    scan_text = mpl_file[mpl_index:]
    prompt = _CONTEXTUAL_RULES_PROMPT_TEMPLATE.format(code=scan_text)
    raw = generate_llm(prompt)
    if not raw or raw.strip().upper() == "NONE":
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return list(sorted(set(str(x).strip().upper() for x in parsed)))
    except Exception:
        # try once more with stricter instruction
        raw2 = generate_llm(prompt + '\nReturn ONLY a valid Python list literal of strings like ["RULE","RULE2"].')
        if not raw2 or raw2.strip().upper() == "NONE":
            return []
        try:
            parsed2 = ast.literal_eval(raw2)
            if isinstance(parsed2, list):
                return list(sorted(set(str(x).strip().upper() for x in parsed2)))
        except Exception:
            return []
    return []

# ---------- Summary generation (full + masked) ----------
BASE_SCREENREADER_PROMPT = (
    'You are a screen reader and came across this data visualization. '
    'Describe it in 1-2 sentences using simple, friendly language. '
    'Mention what kind of visualization it is, its title (if any), any highs and lows, '
    'and what the overall pattern seems to be. Start with "A [visualization type] shows…" '
    'or "A [visualization type] titled [title] shows…" If it is not a data visualization, say "N/A".'
)

def build_summary_prompt(code_snippet: str, withhold_code: str | None) -> str:
    extra = ""
    if withhold_code and withhold_code in RULE_DESCRIPTIONS:
        extra = f"\nCRITICAL: {RULE_DESCRIPTIONS[withhold_code]}"
    return (
        f"{BASE_SCREENREADER_PROMPT}\n"
        f"{extra}\n\n"
        "Here is the Matplotlib code that produced the visualization:\n"
        "```python\n" + code_snippet + "\n```"
    )

def generate_summaries(code_snippet: str, masked_rule: str | None):
    """Returns (summary_full, summary_masked). Empty strings if Gemini disabled."""
    if not HAVE_GEMINI:
        return "", ""
    full_prompt = build_summary_prompt(code_snippet, None)
    masked_prompt = build_summary_prompt(code_snippet, masked_rule)
    s_full = generate_llm(full_prompt)
    s_mask = generate_llm(masked_prompt) if masked_rule else s_full
    # compact to single line for CSV
    s_full = re.sub(r"\s+", " ", s_full).strip()
    s_mask = re.sub(r"\s+", " ", s_mask).strip()
    return s_full, s_mask

def pick_masked_rule(nonctx_list, ctx_list, row_id: int):
    """Pick one violation deterministically (seeded by row_id) from detected ones, else None."""
    candidates = [c for c in (nonctx_list + ctx_list) if c in RULE_DESCRIPTIONS]
    if not candidates:
        return None
    # stable-but-random choice per row
    seed = int(hashlib.md5(str(row_id).encode("utf-8")).hexdigest(), 16)
    rng = random.Random(seed)
    return rng.choice(candidates)

# ---------- Similarity service ----------
def compute_similarity(s1: str, s2: str) -> float | None:
    if not s1 or not s2 or not HAVE_REQUESTS or not SIM_URL:
        return None
    try:
        r = requests.get(SIM_URL, params={"sentence1": s1, "sentence2": s2}, timeout=10)
        if r.ok:
            try:
                # service returns a bare number (float)
                return float(r.json()) if r.headers.get("Content-Type","").startswith("application/json") else float(r.text)
            except Exception:
                return None
        return None
    except Exception:
        return None

# ---------- Load input ----------
fname = SRC if os.path.exists(SRC) else (SRC + '.csv' if os.path.exists(SRC + '.csv') else SRC)
try:
    df = pd.read_csv(fname)
except Exception:
    df = pd.read_csv(SRC + '.csv')
    fname = SRC + '.csv'

print(f"Loaded: {fname}  (rows={len(df)})")

# Detect code column
POSSIBLE_CODE_COLS = ["matplotlib_code","code","file_content","content","snippet","source","body","text"]
code_col = None
for c in POSSIBLE_CODE_COLS:
    if c in df.columns:
        code_col = c; break
if code_col is None:
    for c in df.columns:
        if df[c].dtype == object:
            sample = df[c].dropna().astype(str).head(10)
            if sample.str.contains("import matplotlib|from matplotlib", regex=True, case=False).any():
                code_col = c; break
if code_col is None:
    raise ValueError("Could not find a column with Matplotlib code. Add a column named e.g., 'matplotlib_code'.")

print(f"Using code column: {code_col}")

# ---------- Prepare streaming output ----------
base, ext = os.path.splitext(fname)
out = f"{base}_scored_stream{ext or '.csv'}"

# Build a unified header: row_id + original cols + rule flags + list cols + summaries
orig_cols = list(df.columns)
header_cols = (["row_id"] + orig_cols +
               [c for c in ALL_RULE_COLS if c not in orig_cols] +
               ["NONCONTEXTUAL_LIST","CONTEXTUAL_LIST",
                "SUMMARY_FULL","MASKED_RULE","SUMMARY_MASKED","SUMMARY_SIMILARITY"])

# Resume support: find already processed row_ids
processed_ids = set()
if os.path.exists(out):
    try:
        prev = pd.read_csv(out, usecols=["row_id"])
        processed_ids = set(prev["row_id"].dropna().astype(int).tolist())
        print(f"Resuming; already have {len(processed_ids)} rows in {out}")
    except Exception as e:
        print(f"Note: couldn't read existing {out} for resume ({e}); starting fresh.")

# Write header if new file
if not os.path.exists(out):
    pd.DataFrame(columns=header_cols).to_csv(out, index=False)

# ---------- Row scoring helpers ----------
def score_row(code: str):
    if not isinstance(code, str):
        return [], []
    nonctx = find_noncontextual_flaws(code)
    ctx = find_contextual_flaws(code) if HAVE_GEMINI else []
    return nonctx, ctx

# ---------- Select indices (random sample if requested) ----------
all_indices = df.index.tolist()
if SAMPLE_N and SAMPLE_N > 0 and SAMPLE_N < len(all_indices):
    selected = df.sample(SAMPLE_N, random_state=42).index.tolist()
else:
    selected = all_indices

# ---------- Main loop (streaming) ----------
rows_done = 0
for idx in selected:
    if MAX_ROWS and rows_done >= MAX_ROWS:
        print(f"Hit MAX_ROWS={MAX_ROWS}; stopping early.")
        break

    if int(idx) in processed_ids:
        continue

    row = df.loc[idx]
    row_id = int(idx)
    code = row[code_col]

    # 1) rule detection
    nonctx_list, ctx_list = score_row(code)

    # 2) masked rule (deterministic per-row)
    masked_rule = pick_masked_rule(nonctx_list, ctx_list, row_id)

    # 3) summaries (Gemini may be disabled → empty strings)
    summary_full, summary_masked = generate_summaries(code if isinstance(code, str) else "", masked_rule)

    # 4) similarity (optional; only if we have two summaries and service is up)
    sim = compute_similarity(summary_full, summary_masked)
    if sim is None:
        sim_val = ""
    else:
        # keep a short float for CSV
        sim_val = round(float(sim), 6)

    # 5) Build a single-row dict with original data + flags + lists + summaries
    out_row = dict(row)  # copies original columns
    out_row["row_id"] = row_id
    for r in NONCONTEXTUAL_CODES:
        out_row[r] = 1 if r in nonctx_list else 0
    for r in CONTEXTUAL_CODES:
        out_row[r] = 1 if r in ctx_list else 0
    out_row["NONCONTEXTUAL_LIST"] = ";".join(nonctx_list)
    out_row["CONTEXTUAL_LIST"] = ";".join(ctx_list)
    out_row["SUMMARY_FULL"] = summary_full  # plain text; single line
    out_row["MASKED_RULE"] = masked_rule or ""
    out_row["SUMMARY_MASKED"] = summary_masked  # plain text; single line
    out_row["SUMMARY_SIMILARITY"] = sim_val

    # 6) Append to CSV immediately (no extra prints of summaries)
    pd.DataFrame([out_row])[header_cols].to_csv(out, mode="a", header=False, index=False)

    rows_done += 1
    if rows_done % PRINT_EVERY == 0:
        print(f"Processed {rows_done} rows (last row_id={row_id}) → {out}")

print(f"✅ Done. Appended {rows_done} new rows → {out}")


Loaded: github_matplotlib_audit_1.csv  (rows=14956)
Using code column: code
✅ Done. Appended 10 new rows → github_matplotlib_audit_1_scored_stream.csv


In [11]:
df2 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv")
df2.head()

,row_id,repo,filename,pushed_date,code,MISSING_TITLE,MISSING_XLABEL,MISSING_YLABEL,MISSING_LEGEND,FONTSIZE_TOO_SMALL,...,TRUNCATED_AXES,ASPECT_RATIO_DISTORTION,DUAL_AXES,NON_SEQUENTIAL_AXIS,NONCONTEXTUAL_LIST,CONTEXTUAL_LIST,SUMMARY_FULL,MASKED_RULE,SUMMARY_MASKED,SUMMARY_SIMILARITY
0,3921,safdaraliniazi/Blue-Bank-Loan-Analysis,bank.py,2023-02-10,"# -*- coding: utf-8 -*-\r\n""""""\r\nCreated on W...",1,1,1,0,0,...,0,0,0,0,MISSING_TITLE;MISSING_XLABEL;MISSING_YLABEL,NaN,A bar chart shows the number of people in each...,MISSING_YLABEL,A bar chart shows the number of people in each...,NaN
1,14845,krottapalli1923G/IMDB-Movie-Ratings-Analysis,step5_genre_ratings.py,2025-01-30,import pandas as pd\nimport numpy as np\nimpor...,0,0,0,0,0,...,0,0,0,0,NaN,NaN,A bar chart titled “🎭 Average Rating Per Genre...,NaN,A bar chart titled “🎭 Average Rating Per Genre...,NaN
2,6669,Izzenn/Graphs,PYPT graphs.py,2023-09-24,import matplotlib.pyplot as plt\nimport numpy ...,0,1,1,0,0,...,1,0,0,0,INSUFFICIENT_COLOR_CONTRAST;MISSING_XLABEL;MIS...,TRUNCATED_AXES,NaN,MISSING_XLABEL,A line graph shows two lines with error bars. ...,NaN
3,10315,VSingh1012/DataFy---Data-Research-Project,BarFile.py,2024-06-22,import tkinter as tk\nfrom tkinter import file...,0,1,1,0,0,...,1,0,0,0,INSUFFICIENT_COLOR_CONTRAST;MISSING_XLABEL;MIS...,TRUNCATED_AXES,A bar graph shows the relationship between two...,TRUNCATED_AXES,A bar graph shows the relationship between two...,NaN
4,14872,ulyssetresor13/Medical-Data-Visualizer,medical_data_visualizer.py,2025-01-31,import pandas as pd\nimport seaborn as sns\nim...,1,1,1,1,0,...,0,0,0,0,MISSING_LEGEND;MISSING_TITLE;MISSING_XLABEL;MI...,NaN,NaN,MISSING_LEGEND,A bar chart shows the total counts of differen...,NaN
